# **Traductor**

**Utilzando el dataset de "Combined Data es-Es" entrenar un modelos seq2seq que traduzca del texto de español al ingles**

In [3]:
import pandas as pd

def limpiar_pares(df):
    df = df[["statement", "statement_es_es"]].copy()
    n0 = len(df)

    # 1. Quitar pares con algún lado vacío
    df = df.dropna(subset=["statement", "statement_es_es"])
    df["statement"] = df["statement"].str.strip()
    df["statement_es_es"] = df["statement_es_es"].str.strip()
    df = df[(df["statement"] != "") & (df["statement_es_es"] != "")]
    n1 = len(df)

    # 2. Quitar duplicados exactos del par
    df = df.drop_duplicates(subset=["statement", "statement_es_es"])
    n2 = len(df)

    # 3. Una sola traducción por frase en español (se conserva la primera)
    df = df.drop_duplicates(subset=["statement_es_es"], keep="first")
    n3 = len(df)

    print(f"original:          {n0}")
    print(f"1. sin vacíos:     -{n0 - n1}  -> {n1}")
    print(f"2. sin duplicados: -{n1 - n2}  -> {n2}")
    print(f"3. una por ES:     -{n2 - n3}  -> {n3}")

    return df.reset_index(drop=True)

df = pd.read_csv("Combined Data es-ES.csv", encoding="utf-8")  # ajusta el nombre exacto
df_final = limpiar_pares(df)


original:          53043
1. sin vacíos:     -363  -> 52680
2. sin duplicados: -172  -> 52508
3. una por ES:     -240  -> 52268


In [4]:
df_final.head(1)

,statement,statement_es_es
0,oh my gosh,ay Dios mío


In [5]:
import re

# Limpiar escapes: literal \n, backslash inicial, espacios dobles
def limpia(s):
    return (s.str.replace(r"\\n", " ", regex=True)
             .str.replace(r"^\\", "", regex=True)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())

# Limpiar basura: ° [] {} <> tags HTML
patron_basura = r"° ?\[\]|\[.*?\]|<.*?>|[°{}]"

df_final["statement"] = limpia(df_final["statement"])
df_final["statement_es_es"] = limpia(df_final["statement_es_es"])

df_final["statement"] = df_final["statement"].str.replace(patron_basura, "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
df_final["statement_es_es"] = df_final["statement_es_es"].str.replace(patron_basura, "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

print(f"Limpieza aplicada. Filas: {len(df_final)}")

Limpieza aplicada. Filas: 52268


In [6]:
#  armar text_pairs
text_pairs = []
for i in range(len(df_final)):
    es = df_final["statement_es_es"].iloc[i]
    en = "[start] " + df_final["statement"].iloc[i] + " [end]"
    text_pairs.append((es, en))

print(f"Pares: {len(text_pairs)}")

Pares: 52268


In [7]:
# split 70/15/15

import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

print(f"Train: {len(train_pairs)}")
print(f"Val:   {len(val_pairs)}")
print(f"Test:  {len(test_pairs)}")


Train: 36588
Val:   7840
Test:  7840


In [8]:
# tokenizador


import string
import re
import tensorflow as tf
from keras import layers
import keras

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 20000
sequence_length = 80

spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 2,
    standardize=custom_standardization,
)

train_spanish_texts = [pair[0] for pair in train_pairs]
train_english_texts = [pair[1] for pair in train_pairs]
spanish_tokenizer.adapt(train_spanish_texts)
english_tokenizer.adapt(train_english_texts)

In [9]:
largo_es = sum(1 for pair in text_pairs if len(pair[0].split()) > 80)
largo_en = sum(1 for pair in text_pairs if len(pair[1].split()) > 82)

print(f"Truncadas en español (>80): {largo_es} ({largo_es*100/len(text_pairs):.1f}%)")
print(f"Truncadas en inglés (>82):  {largo_en} ({largo_en*100/len(text_pairs):.1f}%)")

Truncadas en español (>80): 19746 (37.8%)
Truncadas en inglés (>82):  22450 (43.0%)


In [10]:
# Pre-tokenizar y datasets
batch_size = 64

# Pre-tokenizar
train_spa_tokenized = spanish_tokenizer(train_spanish_texts)
train_eng_tokenized = english_tokenizer(train_english_texts)

val_spa_texts = [pair[0] for pair in val_pairs]
val_eng_texts = [pair[1] for pair in val_pairs]
val_spa_tokenized = spanish_tokenizer(val_spa_texts)
val_eng_tokenized = english_tokenizer(val_eng_texts)

def format_dataset(spa_tokens, eng_tokens):
    features = {
        "spanish": spa_tokens,
        "english": eng_tokens[:, :-1]
    }
    labels = eng_tokens[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(spa_tokens, eng_tokens):
    dataset = tf.data.Dataset.from_tensor_slices((spa_tokens, eng_tokens))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_spa_tokenized, train_eng_tokenized)
val_ds = make_dataset(val_spa_tokenized, val_eng_tokenized)

In [11]:
# Modelo (encoder y decoder)

embed_dim = 256
hidden_dim = 1024

# Encoder (español - entrada)
source = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim)(source)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)

# Decoder (inglés - salida)
target = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim)(target)
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

seq2seq_rnn = keras.Model([source, target], target_predictions)

In [12]:
# summary del modelo
seq2seq_rnn.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding (Embedding) │ (None, None, 256) │   5,120,000 │ spanish[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_1           │ (None, None, 256) │   5,120,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 1024)      │   7,876,608 │ embedding[0][0]    │
│ (Bidirectional)       │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_1 (GRU)           │ (None, None,      │   3,938,304 │ embedding_1[0][0], │
│                       │ 1024)             │             │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout (Dropout)     │ (None, None,      │           0 │ gru_1[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense (Dense)         │ (None, None,      │  20,500,000 │ dropout[0][0]      │
│                       │ 20000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 42,554,912 (162.33 MB)

 Trainable params: 42,554,912 (162.33 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
#Compile y fit
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=10, validation_data=val_ds)

Epoch 1/10
572/572 ━━━━━━━━━━━━━━━━━━━━ 444s 773ms/step - accuracy: 0.2214 - loss: 2.9054 - val_accuracy: 0.2422 - val_loss: 2.7581
Epoch 2/10
572/572 ━━━━━━━━━━━━━━━━━━━━ 441s 771ms/step - accuracy: 0.2433 - loss: 2.7242 - val_accuracy: 0.2608 - val_loss: 2.6540
Epoch 3/10
572/572 ━━━━━━━━━━━━━━━━━━━━ 441s 771ms/step - accuracy: 0.2604 - loss: 2.5755 - val_accuracy: 0.2703 - val_loss: 2.5980
Epoch 4/10
572/572 ━━━━━━━━━━━━━━━━━━━━ 441s 771ms/step - accuracy: 0.2756 - loss: 2.4499 - val_accuracy: 0.2771 - val_loss: 2.5666
Epoch 5/10
572/572 ━━━━━━━━━━━━━━━━━━━━ 441s 772ms/step - accuracy: 0.2899 - loss: 2.3407 - val_accuracy: 0.2814 - val_loss: 2.5499
Epoch 6/10
572/572 ━━━━━━━━━━━━━━━━━━━━ 441s 771ms/step - accuracy: 0.3042 - loss: 2.2470 - val_accuracy: 0.2861 - val_loss: 2.5388
Epoch 7/10
572/572 ━━━━━━━━━━━━━━━━━━━━ 441s 772ms/step - accuracy: 0.3185 - loss: 2.1653 - val_accuracy: 0.2864 - val_loss: 2.5496
Epoch 8/10
572/572 ━━━━━━━━━━━━━━━━━━━━ 441s 771ms/step - accuracy: 0.3315 -

In [20]:
# traducción

import numpy as np


eng_vocab = english_tokenizer.get_vocabulary()
eng_index_lookup = dict(zip(range(len(eng_vocab)), eng_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = spanish_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = english_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = eng_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

# Filtrar frases cortas
test_cortas = [pair for pair in test_pairs if len(pair[0].split()) <= 20]
muestra = random.sample(test_cortas, 5)

for es, en in muestra:
    pred = generate_translation(es)
    print("-")
    print(f"Español:    {es}")
    print(f"Predicción: {pred}")
    print(f"Referencia: {en}")


-
Español:    Estoy bailando en el borde del inquietud.
Predicción: [start] im in the middle of the class [end]
Referencia: [start] I'm dancing on the edge of restlessness [end]
-
Español:    joder, de verdad solo tengo 4 horas para empezar y terminar este trabajo para mañana me odio
Predicción: [start] damn fucking damn i have no fucking fucking [UNK] i have a job tomorrow and i hate myself i hate this shit [end]
Referencia: [start] damn i really only have 4 hours to start and finish this paper for tomorrow i hate myself [end]
-
Español:    ¡Quédate con abusos mentales como él! ¡Los abusos mentales y emocionales por: SHAWN CLIFTON KING
Predicción: [start] [UNK] with family members [UNK] [UNK] [UNK] and [UNK] [UNK] [end]
Referencia: [start] Mental an emotional abuses like him should be fucking shot! Mental and emotional abuse by: SHAWN CLIFTON KING [end]
-
Español:    Eso arruinaría todo, me gustaría tanto ir.
Predicción: [start] that will be really good [end]
Referencia: [start] that 

In [21]:
!pip install nltk rouge-score -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [23]:
# traducciones test

test_spa_texts = [pair[0] for pair in test_pairs]
test_eng_texts = [pair[1] for pair in test_pairs]

# Usamos una muestra
sample_size = 500
indices = random.sample(range(len(test_pairs)), sample_size)

referencias = []
predicciones = []
for idx in indices:
    ref = test_eng_texts[idx].replace("[start] ", "").replace(" [end]", "")
    pred = generate_translation(test_spa_texts[idx]).replace("[start] ", "").replace(" [end]", "")
    referencias.append(ref)
    predicciones.append(pred)

print(f"Traducciones generadas: {len(predicciones)}")

Traducciones generadas: 500


In [25]:
# bleu score

from nltk.translate.bleu_score import corpus_bleu

refs_bleu = [[ref.split()] for ref in referencias]
preds_bleu = [pred.split() for pred in predicciones]

bleu = corpus_bleu(refs_bleu, preds_bleu)
print(f"BLEU score: {bleu:.4f}")

BLEU score: 0.0149


In [26]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

r1, r2, rl = 0, 0, 0
for ref, pred in zip(referencias, predicciones):
    scores = scorer.score(ref, pred)
    r1 += scores["rouge1"].fmeasure
    r2 += scores["rouge2"].fmeasure
    rl += scores["rougeL"].fmeasure

n = len(referencias)
print(f"ROUGE-1: {r1/n:.4f}")
print(f"ROUGE-2: {r2/n:.4f}")
print(f"ROUGE-L: {rl/n:.4f}")

ROUGE-1: 0.3332
ROUGE-2: 0.1060
ROUGE-L: 0.2452


In [27]:
def word_f1(ref, pred):
    ref_words = set(ref.split())
    pred_words = set(pred.split())
    common = ref_words & pred_words
    if len(pred_words) == 0:
        precision = 0
    else:
        precision = len(common) / len(pred_words)
    if len(ref_words) == 0:
        recall = 0
    else:
        recall = len(common) / len(ref_words)
    if precision + recall == 0:
        return 0, 0, 0
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

total_p, total_r, total_f1 = 0, 0, 0
for ref, pred in zip(referencias, predicciones):
    p, r, f = word_f1(ref, pred)
    total_p += p
    total_r += r
    total_f1 += f

n = len(referencias)
print(f"Precision: {total_p/n:.4f}")
print(f"Recall:    {total_r/n:.4f}")
print(f"F1 score:  {total_f1/n:.4f}")

Precision: 0.4159
Recall:    0.2160
F1 score:  0.2674
